# 🔴 Interview: Tensor Parallel MLP

Megatron-style column-parallel + row-parallel MLP

In [ ]:
import torch
import torch.nn as nn


In [ ]:
# ✅ INTERVIEW

class TensorParallelMLP(nn.Module):
    """
    手写张量并行 MLP —— 面试版。

    面试考点:
    1. 列并行 (Column Parallel): W1 按列切，每个设备算部分隐藏维度
    2. 行并行 (Row Parallel): W2 按行切，每个设备产生完整输出
    3. All-reduce: 所有分片输出求和
    4. 为什么这样切？列并行和行并行配合只需一次 all-reduce

    参数: d_model, d_ff, world_size
    """
    def __init__(self, d_model, d_ff, world_size):
        super().__init__()
        self.world_size = world_size
        shard = d_ff // world_size

        # W1 列分片: 每片 (d_model, shard)
        self.w1_shards = nn.ParameterList([
            nn.Parameter(torch.randn(d_model, shard) * (2 / d_model) ** 0.5)
            for _ in range(world_size)
        ])
        # W2 行分片: 每片 (shard, d_model)
        self.w2_shards = nn.ParameterList([
            nn.Parameter(torch.randn(shard, d_model) * (2 / d_ff) ** 0.5)
            for _ in range(world_size)
        ])

    def forward(self, x):
        """
        x: (B, d_model) → (B, d_model)

        每个分片 i:
          h_i = GELU(x @ w1_i)    # (B, shard)
          p_i = h_i @ w2_i        # (B, d_model)
        output = sum(p_i)          # all-reduce
        """
        output = None
        for w1, w2 in zip(self.w1_shards, self.w2_shards):
            # 列并行: x @ w1 只算部分隐藏维度
            h = torch.nn.functional.gelu(x @ w1)  # (B, shard)
            # 行并行: h @ w2 产生完整输出
            partial = h @ w2                       # (B, d_model)
            # All-reduce: 累加
            output = partial if output is None else output + partial
        return output

In [ ]:
# Verify: compare TensorParallelMLP against a standard single-GPU MLP
torch.manual_seed(42)
d_model, d_ff, world_size = 64, 256, 4
x = torch.randn(2, d_model)

tp_mlp = TensorParallelMLP(d_model, d_ff, world_size)
out = tp_mlp(x)
print("Output shape:", out.shape)  # expect (2, 64)

# Build equivalent full MLP from the sharded weights
w1_full = torch.cat([p.data for p in tp_mlp.w1_shards], dim=1)  # (d_model, d_ff)
w2_full = torch.cat([p.data for p in tp_mlp.w2_shards], dim=0)  # (d_ff, d_model)
ref = torch.nn.functional.gelu(x @ w1_full) @ w2_full

print("Max diff vs reference:", (out - ref).abs().max().item())  # expect ~0

In [ ]:
from torch_judge import check
check("tensor_parallel")

## 📝 核心思路总结

1. **列并行 + 行并行**：W1 按列切（每个设备算部分隐藏维度），W2 按行切（每个设备接收部分输入），两者配合只需一次 all-reduce。
2. **数学等价性**：分片计算的结果与完整矩阵计算完全一致，因为 `sum(h_i @ w2_i) = h @ W2`（矩阵分块乘法的性质）。
3. **All-reduce 是唯一的通信**：列并行的输出天然分片，行并行的输出需要求和，所以整个前向传播只有一次 all-reduce。
4. **GELU 在中间**：激活函数必须在列并行之后、行并行之前，因为每个分片独立计算自己的隐藏维度。